In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, to_timestamp, explode, row_number
from pyspark.sql.window import Window

# 1. Setup Variables
catalog = "maritime_ais"
bronze_schema = "maritime_bronze"
silver_schema = "maritime_silver"
checkpoint_base = "abfss://maritime-lake@maritimepipeline.dfs.core.windows.net/checkpoints/silver"

# 2. Define Upsert Logic (Spark Connect safe + dedup + null-key filter)
def upsert_to_silver(microBatchDF, batchId, table_name, merge_condition, dedup_keys=None):
    try:
        _spark = microBatchDF.sparkSession

        # Filter out rows with null keys to prevent merge crashes
        clean_df = microBatchDF.filter(col("site_number").isNotNull() & col("last_update").isNotNull())

        if dedup_keys:
            w = Window.partitionBy(*dedup_keys).orderBy(col(dedup_keys[0]))
            clean_df = (
                clean_df.withColumn("_rn", row_number().over(w))
                .filter(col("_rn") == 1)
                .drop("_rn")
            )

        if not _spark.catalog.tableExists(table_name):
            clean_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
            return

        delta_table = DeltaTable.forName(_spark, table_name)
        delta_table.alias("target").merge(
            clean_df.alias("source"),
            merge_condition
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    except Exception as e:
        print(f"CRITICAL ERROR in batch {batchId}: {str(e)}")
        raise e

# 3. Process Sea State
print("Processing Silver Sea State...")
df_bronze_sea = spark.readStream.table(f"{catalog}.{bronze_schema}.sea_state")

df_exploded_sea = df_bronze_sea.select(explode(col("features")).alias("feature"))

df_silver_sea = df_exploded_sea.select(
    col("feature.properties.siteNumber").cast("string").alias("site_number"),
    col("feature.properties.siteName").alias("site_name"),
    col("feature.properties.siteType").alias("site_type"),
    col("feature.geometry.coordinates").getItem(0).cast("double").alias("longitude"),
    col("feature.geometry.coordinates").getItem(1).cast("double").alias("latitude"),
    col("feature.properties.seaState").alias("sea_state"),
    col("feature.properties.temperature").cast("double").alias("temperature"),
    to_timestamp(col("feature.properties.lastUpdate")).alias("last_update")
)

sea_table = f"{catalog}.{silver_schema}.sea_state"

# Composite key to maintain historical sensor readings while handling exact duplicates
composite_merge_condition = "target.site_number = source.site_number AND target.last_update = source.last_update"

query_sea = (
    df_silver_sea.writeStream
    .foreachBatch(lambda df, epoch_id: upsert_to_silver(
        df, epoch_id, sea_table, composite_merge_condition, dedup_keys=["site_number", "last_update"]
    ))
    .option("checkpointLocation", f"{checkpoint_base}/sea_state")
    .trigger(availableNow=True)
    .start()
)
query_sea.processAllAvailable()
print("Completed Sea State Processing.")